# Etykietowanie danych

W tym notatniku będziesz etykietować dane - innymi słowy, powiesz AI, co znajduje się na obrazach, aby mogło nauczyć się to rozpoznawać!

## O notatniku Jupyter

To jest notatnik Jupyter. Zawiera różne *komórki*.

Komórki mają różne typy. Na przykład ten tekst jest zapisany w komórce Markdown.

Poniżej znajduje się komórka z kodem:

In [ ]:
print("Witaj z notatnika Jupyter!")

Możesz uruchomić komórkę, ustawiając w niej kursor i robiąc jedną z poniższych rzeczy:
1. Klikając przycisk uruchamiania u góry ekranu
2. Naciskając ctrl-enter
3. Naciskając shift-enter, co wykona bieżącą komórkę i przejdzie do następnej

Aby ukończyć warsztat, przejdź przez każdą komórkę, spróbuj zrozumieć, co robi, i ją uruchom!

## Importy

Najpierw importujemy potrzebne biblioteki Pythona

In [ ]:
import ipywidgets as widgets
from pathlib import Path

# Obsługa plików

Etykietowanie często wykonuje się w wyspecjalizowanych aplikacjach. W naszym przypadku użyjemy zamiast tego prostego programu w Pythonie. Najpierw musimy powiedzieć programowi, gdzie ma szukać danych.

In [ ]:
raw_data_dir = Path("raw_data")  # Katalog z surowymi danymi
forward_data_dir = Path("labeled_data/forward")  # Zdjęcia z wolną drogą na wprost trafią tutaj
obstacle_data_dir = Path("labeled_data/obstacle")  # Zdjęcia z przeszkodami trafią tutaj

In [ ]:
# Utwórz katalogi, jeśli nie istnieją
forward_data_dir.mkdir(exist_ok=True)
obstacle_data_dir.mkdir(exist_ok=True)

# Funkcja pobierająca kolejne zdjęcie z katalogu surowych danych
def get_next_photo():
  try:
    return next(raw_data_dir.iterdir()).name
  except StopIteration:
    return None

curr_photo = get_next_photo()

# Policz, ile mamy zdjęć łącznie i ile jest już oznaczonych
total_photo_count = len(list(raw_data_dir.iterdir())) + len(list(forward_data_dir.iterdir())) + len(list(obstacle_data_dir.iterdir()))
photos_labeled = total_photo_count - len(list(raw_data_dir.iterdir()))

print(f"Utworzone katalogi: {forward_data_dir}, {obstacle_data_dir}")
print(f"Liczba wszystkich zdjęć: {total_photo_count}")
print(f"Liczba oznaczonych zdjęć: {photos_labeled}")

Możesz zauważyć, że część zdjęć na starcie jest już oznaczona. Ułatwiliśmy Ci pracę, żebyś nie musiał/a oznaczać kilkuset zdjęć.

## Interfejs etykietowania

Ta długa komórka tworzy interfejs dla naszego procesu etykietowania. Najpierw spróbuj ją uruchomić i zobacz, co się stanie!

In [ ]:
# Konfiguracja widżetów
# Widżet do wyświetlania bieżącego zdjęcia
image_widget = widgets.Image(
  value=b"",
  format="jpg",
  width=600,
  height=600,
)

# Nagłówek pokazujący numer bieżącego zdjęcia
header = widgets.HTML(
  value=f"",
  layout=widgets.Layout(margin='10px 0px 20px 0px')
)

# Przyciski do oznaczania zdjęć z wolną drogą
forward_button = widgets.Button(
  description="Droga wolna, jedź prosto!",
  layout=widgets.Layout(width='300px', height='100px'),
  button_style="success",
  style={'font_weight': 'bold', 'font_size': '20px'}
)

# Przycisk do oznaczania zdjęć z przeszkodami
obstacle_button = widgets.Button(
  description="Wykryto przeszkodę!",
  layout=widgets.Layout(width='300px',height='100px'),
  button_style="danger",
  style={'font_weight': 'bold', 'font_size': '20px'}
)

# Ułóż widżety
hbox = widgets.HBox([forward_button, obstacle_button])
vbox = widgets.VBox([header, image_widget, hbox])

# Po kliknięciu przycisku przenieś bieżące zdjęcie do odpowiedniego katalogu i odśwież widok
def update_image():
  global curr_photo
  curr_photo = get_next_photo()

  global photos_labeled
  photos_labeled = total_photo_count - len(list(raw_data_dir.iterdir()))

  # Zaktualizuj nagłówek i widżet obrazu
  if curr_photo:
    header.value = f"<h2>Zdjęcie {photos_labeled + 1}/{total_photo_count}</h2>"
    image_widget.value = (raw_data_dir / curr_photo).read_bytes()
  else:
    header.value = "<h2>Wszystkie zdjęcia oznaczone!</h2>"
    vbox.children = [header]
    image_widget.value = b""

# Obsługa kliknięć przycisków
def on_forward_button_clicked(_b):
  (raw_data_dir / curr_photo).rename(forward_data_dir / curr_photo)
  print("Przeniesiono do forward_data")
  update_image()

def on_obstacle_button_clicked(_b):
  (raw_data_dir / curr_photo).rename(obstacle_data_dir / curr_photo)
  print("Przeniesiono do obstacle_data")
  update_image()

forward_button.on_click(on_forward_button_clicked)
obstacle_button.on_click(on_obstacle_button_clicked)

update_image()

display(vbox)

Powyżej widzisz zdjęcie i dwa przyciski. Jeśli uważasz, że to zdjęcie przedstawia wolną drogę i robot powinien jechać prosto, naciśnij zielony przycisk. Jeśli uważasz, że droga jest zablokowana i robot powinien skręcić, naciśnij czerwony przycisk. Zastanów się dobrze, ponieważ to jedna z najważniejszych części trenowania naszego modelu AI!

Gdy oznaczysz wszystkie obrazy, przejdź do kolejnego notatnika - `2_pl_model_training.ipynb`